# Phase 9: Granular Inference and Insight Aggregation
This notebook implements **Phase 9** using a subset of `DB_master_reviews_combined.csv` and the fine-tuned **XLM-RoBERTa (Weighted)** model:
- **Step 9.1: Targeted Inference** — Decomposes reviews into sentence-aspect pairs and predicts aspect-level sentiment.
- **Step 9.2: Sentiment Aggregation** — Compiles polarity counts and prevailing sentiment percentages across operational dimensions.
- **Step 9.3: Output Structuring (Granular System Interface)** — Formats a directory-style comparison of establishments replacing vague star ratings with granular aspect scores.

In [1]:
import random
import re
import numpy as np
import pandas as pd
import torch
from nltk.tokenize import sent_tokenize
from transformers import AutoTokenizer, XLMRobertaForSequenceClassification

# 1. Enforce Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [3]:
# 2. Load Dataset & Integrity Checks
raw_df = pd.read_csv('../Dataset/DB_master_reviews_combined.csv')
print(f'Raw dataset rows: {len(raw_df)} | Missing review_text: {raw_df["review_text"].isnull().sum()}')

# Drop missing review text and exact duplicate records
clean_df = raw_df.dropna(subset=['review_text', 'place_name', 'rating']).drop_duplicates().copy()
assert clean_df[['place_name', 'rating', 'review_text']].isnull().sum().sum() == 0, 'Null values remain!'

# Take a small subset: sample 50 reviews each from 5 well-reviewed coffee & matcha establishments
target_shops = clean_df['place_name'].value_counts().head(5).index.tolist()
subset_df = (
    clean_df[clean_df['place_name'].isin(target_shops)]
    .groupby('place_name')
    .sample(n=50, random_state=SEED)
    .reset_index(drop=True)
)
print(f'Sampled subset: {len(subset_df)} reviews across {subset_df["place_name"].nunique()} shops.')

Raw dataset rows: 19586 | Missing review_text: 7015
Sampled subset: 250 reviews across 5 shops.


In [4]:
# 3. Step 9.1: Targeted Inference
aspect_dictionary = {
    'Product Quality': [
        'coffee', 'matcha', 'latte', 'espresso', 'sikwate', 'barako', 'pandesal',
        'pastry', 'cake', 'taste', 'lami', 'masarap', 'lasa', 'tam-is', 'umay',
        'lamiay', 'food', 'drinks', 'lasaw', 'tab-ang', 'init', 'bugnaw'
    ],
    'Customer Service': [
        'barista', 'staff', 'crew', 'cashier', 'service', 'mabait', 'accommodating',
        'paspas', 'dugay', 'maayos', 'attitude', 'buotan', 'snob', 'friendly'
    ],
    'Ambiance and Atmosphere': [
        'ambiance', 'vibe', 'aesthetic', 'chill', 'quiet', 'noisy', 'saba', 'music',
        'muni-muni', 'chika', 'tropa', 'atmosphere', 'crowded', 'ig-worthy',
        'instagrammable', 'tambay'
    ],
    'Facilities and Amenities': [
        'wifi', 'internet', 'connection', 'socket', 'outlet', 'plug', 'cr', 'restroom',
        'comfort room', 'aircon', 'tables', 'chairs', 'space', 'study', 'tuon', 'work',
        'laptop', 'kusog', 'igang'
    ],
    'Price and Value': [
        'price', 'value', 'mahal', 'barato', 'sulit', 'tipid', 'overpriced',
        'affordable', 'sayang', 'budget', 'expensive', 'cheap', 'pricy'
    ],
    'Store Operations and Accessibility': [
        'parking', 'location', 'accessible', 'layo', 'malayo', 'duol', 'hours', 'open',
        'close', 'traffic'
    ]
}

compiled_patterns = {
    aspect: re.compile(r'\b(?:' + '|'.join(map(re.escape, kw)) + r')\b')
    for aspect, kw in aspect_dictionary.items()
}

def extract_aspects(sentence):
    if not isinstance(sentence, str):
        return []
    s_low = sentence.lower()
    return [aspect for aspect, pat in compiled_patterns.items() if pat.search(s_low)]

subset_df['sentence'] = subset_df['review_text'].astype(str).apply(sent_tokenize)
exploded_df = subset_df.explode('sentence').dropna(subset=['sentence']).reset_index(drop=True)
exploded_df['TargetAspect'] = exploded_df['sentence'].apply(extract_aspects)
aspect_pairs_df = exploded_df[exploded_df['TargetAspect'].astype(bool)].explode('TargetAspect').reset_index(drop=True)

# Load Fine-Tuned Weighted XLM-RoBERTa Model
MODEL_DIR = '../_7_Model Fine-Tuning/xlmr_absa_weighted'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = {0: 'POSITIVE', 1: 'NEUTRAL', 2: 'NEGATIVE'}
predictions = []
batch_size = 32

with torch.no_grad():
    for i in range(0, len(aspect_pairs_df), batch_size):
        batch = aspect_pairs_df.iloc[i:i + batch_size]
        enc = tokenizer(
            batch['TargetAspect'].astype(str).tolist(),
            batch['sentence'].astype(str).tolist(),
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)
        logits = model(**enc).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        predictions.extend([id2label[p] for p in preds])

aspect_pairs_df['Predicted_Sentiment'] = predictions
print(f'Total sentence-aspect pairs inferred: {len(aspect_pairs_df)}')
aspect_pairs_df[['place_name', 'Shop_Type', 'rating', 'TargetAspect', 'sentence', 'Predicted_Sentiment']].head(5)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total sentence-aspect pairs inferred: 320


,place_name,Shop_Type,rating,TargetAspect,sentence,Predicted_Sentiment
0,BRGR Alley Café,Coffee,5,Price and Value,Sulit na siya!!!!,NEGATIVE
1,BRGR Alley Café,Coffee,5,Store Operations and Accessibility,++ big parking sooo 5 ⭐️,POSITIVE
2,BRGR Alley Café,Coffee,5,Product Quality,Great place and great food.,POSITIVE
3,BRGR Alley Café,Coffee,5,Product Quality,"The ambiance is nice, friendly staff and delic...",POSITIVE
4,BRGR Alley Café,Coffee,5,Customer Service,"The ambiance is nice, friendly staff and delic...",POSITIVE


In [4]:
# 4. Step 9.2: Sentiment Aggregation (Overall Operational Dimensions)
aspect_counts = pd.crosstab(aspect_pairs_df['TargetAspect'], aspect_pairs_df['Predicted_Sentiment'])
for col in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    if col not in aspect_counts.columns:
        aspect_counts[col] = 0

aspect_counts = aspect_counts[['POSITIVE', 'NEUTRAL', 'NEGATIVE']]
aspect_pct = aspect_counts.div(aspect_counts.sum(axis=1), axis=0).mul(100).round(1)

dimension_summary = pd.DataFrame({
    'Total Mentions': aspect_counts.sum(axis=1),
    'Positive Count': aspect_counts['POSITIVE'],
    'Neutral Count': aspect_counts['NEUTRAL'],
    'Negative Count': aspect_counts['NEGATIVE'],
    '% Positive': aspect_pct['POSITIVE'].astype(str) + '%',
    '% Neutral': aspect_pct['NEUTRAL'].astype(str) + '%',
    '% Negative': aspect_pct['NEGATIVE'].astype(str) + '%'
}).sort_values(by='Total Mentions', ascending=False)

dimension_summary

,Total Mentions,Positive Count,Neutral Count,Negative Count,% Positive,% Neutral,% Negative
TargetAspect,,,,,,,
Product Quality,120,100,12,8,83.3%,10.0%,6.7%
Customer Service,97,85,4,8,87.6%,4.1%,8.2%
Ambiance and Atmosphere,55,52,0,3,94.5%,0.0%,5.5%
Facilities and Amenities,20,12,2,6,60.0%,10.0%,30.0%
Price and Value,18,12,0,6,66.7%,0.0%,33.3%
Store Operations and Accessibility,10,8,0,2,80.0%,0.0%,20.0%


In [5]:
# 5. Step 9.3: Output Structuring (Granular System Interface - Directory Listing)
shop_aspect_group = (
    aspect_pairs_df.groupby(['place_name', 'TargetAspect'])['Predicted_Sentiment']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .mul(100)
    .round(0)
)

for col in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    if col not in shop_aspect_group.columns:
        shop_aspect_group[col] = 0.0

# Format prevailing sentiment label per shop-aspect cell (e.g., '85% Positive' or '60% Negative')
def format_aspect_score(row):
    pos, neu, neg = int(row['POSITIVE']), int(row['NEUTRAL']), int(row['NEGATIVE'])
    if pos >= neg and pos >= neu:
        return f'{pos}% Positive'
    elif neg >= pos and neg >= neu:
        return f'{neg}% Negative'
    return f'{neu}% Neutral'

shop_aspect_group['Formatted_Score'] = shop_aspect_group.apply(format_aspect_score, axis=1)
directory_pivot = shop_aspect_group['Formatted_Score'].unstack(fill_value='No Mentions')

# Merge with traditional average star rating & shop type for side-by-side comparison
shop_meta = subset_df.groupby('place_name').agg(
    Shop_Type=('Shop_Type', 'first'),
    Avg_Star_Rating=('rating', lambda x: f'{x.mean():.1f} Stars')
)

granular_directory_df = shop_meta.join(directory_pivot).reset_index()
granular_directory_df.head(5)

,place_name,Shop_Type,Avg_Star_Rating,Ambiance and Atmosphere,Customer Service,Facilities and Amenities,Price and Value,Product Quality,Store Operations and Accessibility
0,BRGR Alley Café,Coffee,4.8 Stars,100% Positive,84% Positive,100% Positive,60% Negative,93% Positive,100% Positive
1,Cafe Demitasse,Coffee,4.4 Stars,82% Positive,88% Positive,100% Positive,100% Positive,81% Positive,100% Positive
2,Starbucks,Coffee,4.5 Stars,86% Positive,88% Positive,67% Negative,No Mentions,81% Positive,100% Positive
3,Starbucks - SM City Davao,Coffee,4.3 Stars,100% Positive,88% Positive,50% Positive,100% Negative,64% Positive,50% Positive
4,The Coffee Bar at GMall of Davao,Coffee,4.6 Stars,100% Positive,92% Positive,100% Positive,67% Positive,88% Positive,No Mentions


In [5]:
brgr_pv_sample = aspect_pairs_df[                                                                                                                             
    (aspect_pairs_df['place_name'].str.contains('BRGR Alley', case=False, na=False)) &                                                                        
    (aspect_pairs_df['TargetAspect'] == 'Price and Value')                                                                                                    
]                                                                                                                                                             
                                                                                                                                                                
print(f"Row count: {len(brgr_pv_sample)}")                                                                                                                    
display(brgr_pv_sample[['place_name', 'sentence', 'Predicted_Sentiment']])                                                                                    
                                                                                                                                                                  


Row count: 5


,place_name,sentence,Predicted_Sentiment
0,BRGR Alley Café,Sulit na siya!!!!,NEGATIVE
18,BRGR Alley Café,The price is good for these delicious foods.,POSITIVE
36,BRGR Alley Café,"Price-wise, it feels slightly on the higher si...",NEGATIVE
65,BRGR Alley Café,Price aint that bad either.,NEGATIVE
78,BRGR Alley Café,"The food is super sulit, big serving and yummy!",POSITIVE
